# 05 — Evaluation
Loads the trained model from `data/models/ecg_cnn.pt` and evaluates it on the held-out validation set.
Generates classification report, per-class AUC, and confusion matrices.

In [ ]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os, pickle
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, roc_auc_score,
                              multilabel_confusion_matrix, ConfusionMatrixDisplay)
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: LOAD DATA & MODEL
# ============================================================
SAVE_DIR  = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'data', 'models')

X = np.load(os.path.join(SAVE_DIR, 'X_clean.npy'))
y = np.load(os.path.join(SAVE_DIR, 'y.npy'))

with open(os.path.join(SAVE_DIR, 'classes.pkl'), 'rb') as f:
    CLASSES = pickle.load(f)

# Recreate same val split as training
X_t = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)
y_t = torch.tensor(y, dtype=torch.float32)
_, X_val, _, y_val = train_test_split(X_t, y_t, test_size=0.2, random_state=42)

val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

# Load model
class ECG_CNN(nn.Module):
    def __init__(self, n_leads=12, n_classes=5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_leads, 32, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),      nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),     nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

checkpoint = torch.load(os.path.join(MODEL_DIR, 'ecg_cnn.pt'), map_location=DEVICE)
model = ECG_CNN(n_leads=12, n_classes=len(CLASSES)).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded ✅")

In [ ]:
# ============================================================
# CELL 3: RUN INFERENCE ON VALIDATION SET
# ============================================================

all_logits, all_labels = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        all_logits.append(model(xb).cpu().numpy())
        all_labels.append(yb.numpy())

logits = np.concatenate(all_logits)
labels = np.concatenate(all_labels)

probs  = torch.sigmoid(torch.tensor(logits)).numpy()
preds  = (probs >= 0.5).astype(int)

print(f"Validation samples: {len(labels)}")

In [ ]:
# ============================================================
# CELL 4: CLASSIFICATION REPORT
# ============================================================

print(classification_report(labels, preds, target_names=CLASSES, zero_division=0))

In [ ]:
# ============================================================
# CELL 5: PER-CLASS ROC-AUC
# ============================================================

print(f"{'Class':<8} {'AUC':>6}")
print('-' * 18)
for i, cls in enumerate(CLASSES):
    try:
        auc = roc_auc_score(labels[:, i], probs[:, i])
        print(f"{cls:<8} {auc:>6.4f}")
    except Exception:
        print(f"{cls:<8}    N/A")

In [ ]:
# ============================================================
# CELL 6: CONFUSION MATRICES PER CLASS
# ============================================================

mcm = multilabel_confusion_matrix(labels, preds)
fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 3))

for ax, matrix, cls in zip(axes, mcm, CLASSES):
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix,
                                   display_labels=['Neg', 'Pos'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(cls, fontweight='bold')

fig.suptitle('Per-Class Confusion Matrices', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()